# Accel event sim

Thin front-end over the `accel` package (physics lives there; see `accel/sim.py` docstring).
Compares three runs: **ideal TC** (quasi-static, force = min(motor, peak grip) — the old joe-accel model),
**no TC**, and **TC** (slip-target PI in `accel/control.py`).

Legacy version: `accel_weight_transfer.ipynb`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from accel import Car, CurveTire, MotorMap, PacejkaTire, SlipTC, quasi_static, simulate

plt.style.use("bmh")
motor = MotorMap("228HV")
tire = CurveTire(mu_peak=1.4)          # or PacejkaTire() for the TTC LC0 fit
car = Car(final_drive=3.5)
tc = SlipTC.for_tire(tire)

runs = [quasi_static(car, motor, tire=tire), simulate(car, motor, tire=tire),
        simulate(car, motor, tire=tire, tc=tc)]
pd.DataFrame([r.summary() for r in runs])

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 15), sharex=True)
for r in runs:
    d = r.df
    axes[0].plot(d.t, d.v * 3.6, label=r.label)
    axes[1].plot(d.t, d.a / 9.81, label=r.label)
    axes[2].plot(d.t, d.motor_torque, label=r.label)
    axes[4].plot(d.t, d.bus_v, label=r.label)
for r in runs[1:]:
    axes[3].plot(r.df.t, r.df.slip.clip(upper=1.0), label=r.label)
axes[3].axhline(tc.target_slip, color="r", ls="--", label="TC target")
for ax, y in zip(axes, ["speed (km/h)", "accel (g)", "motor torque (Nm)", "slip ratio", "bus voltage (V)"]):
    ax.set_ylabel(y); ax.legend()
axes[-1].set_xlabel("time (s)")
plt.tight_layout()

## Launch detail: what TC is doing

In [ ]:
d = runs[2].df.query("t < 1.0")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(d.t, d.slip, label="measured slip (TC input)")
ax1.plot(d.t, d.kappa, label="tire slip (transient)")
ax1.axhline(tc.target_slip, color="r", ls="--")
ax1.set_ylim(-0.1, 0.6); ax1.legend()
ax2.plot(d.t, d.tc_scale, label="TC torque scale")
ax2.plot(d.t, d.fz_rear / d.fz_rear.max(), label="rear Fz (normalized)")
ax2.legend(); ax2.set_xlabel("time (s)")
plt.tight_layout()

## Final drive sweep

In [ ]:
rows = []
for fd in [2.4, 2.8, 3.2, 3.6, 4.0, 4.5, 5.0]:
    c = Car(final_drive=fd)
    for r in [quasi_static(c, motor, tire=tire), simulate(c, motor, tire=tire),
              simulate(c, motor, tire=tire, tc=tc)]:
        rows.append({"fd": fd, "label": r.label, "time": r.accel_time})
sweep = pd.DataFrame(rows).pivot(index="fd", columns="label", values="time")
ax = sweep.plot(marker="o", figsize=(10, 5), ylabel="accel time (s)", title=f"{motor.name}, {type(tire).__name__}")
sweep